# Venusian PCW Detector · 数据与模型工作流

在这里运行缓存生成、事件分析、模型训练、固定窗口评价和连续搜索。Notebook 调用项目的既有实现，使用同一套输入协议、检查点和输出目录。

安装 `notebook` 可选依赖并选用安装了项目的 Python 内核。默认执行输入校验和参考分数评价。启用某项计算前，修改下面对应的开关和输入路径。绘图在 [figures.ipynb](figures.ipynb) 进行。

In [ ]:
# 数据目录。None 表示沿用同名环境变量。
MAG_DIR = None  # VENUS_EXPRESS_RAW_DATA_DIR
FROZEN_FEATURE_DIR = None  # VENUS_EXPRESS_TRAINING_INPUT_DIR
EVENTS_CSV = None  # None 使用仓库附带的事件清单
DEVICE = "cpu"  # cpu、mps 或 cuda
WORKERS = 2

# 日特征和目录分析。None 表示处理完整清单。
RUN_CACHE = False
RUN_CATALOGUE = False
RUN_PSD_AUDIT = False
EVENT_DAYS = ["2006-04-24"]

# 训练集和验证集。可选 MSeed、MExtend1、MExtend2、MStandard、holdout2013。
STAGE = "MStandard"
TRAIN_SEEDS = [2026]
TRAIN_INPUT = "bam"  # bam 或 frozen
PREPARE_TRAINING_WINDOWS = False
RUN_TRAIN = False

# 固定窗口评价。cached 用归档分数；frozen 和 bam 用参考检查点重新评分。
EVALUATION_INPUT = "cached"  # cached、frozen、bam 或 None
PREPARE_EVALUATION_WINDOWS = False  # 为四阶段重建验证集及固定测试集
EVAL_TRAINED_SEED = None  # 例如 2026，额外评价刚训练或已有的本地检查点

# 连续搜索与曝光。SEARCH_DAYS = None 会扫描该范围全部观测日。
RUN_SEARCH = False
SEARCH_SCOPE = "mission"  # mission 或 holdout
SEARCH_DAYS = ["2007-02-04"]  # holdout 可用 ["2013-11-21"]
RESUME_SEARCH = False
RUN_EXPOSURE = False

## 1. 校验输入

In [ ]:
import os
from pathlib import Path
import shlex
import subprocess
import sys

import pandas as pd
from IPython.display import display

from venus_express.repro.settings import BUILD, INPUTS, ROOT, display_path, read, redact_output, write

for name, path in (
    ("VENUS_EXPRESS_RAW_DATA_DIR", MAG_DIR),
    ("VENUS_EXPRESS_TRAINING_INPUT_DIR", FROZEN_FEATURE_DIR),
):
    if path is not None:
        os.environ[name] = str(Path(path).expanduser().resolve())

os.environ.setdefault("MPLCONFIGDIR", str(BUILD / "matplotlib"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

def run_cli(*args):
    command = [sys.executable, "-u", "-m", "venusian_pcw", *map(str, args)]
    shown = shlex.join(["venusian-pcw", *(display_path(arg) if isinstance(arg, Path) else str(arg) for arg in args)])
    print(shown, flush=True)
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                print(redact_output(line, *args), end="", flush=True)
        except KeyboardInterrupt:
            process.terminate()
            raise
        status = process.wait()
    if status:
        raise RuntimeError(f"{shown} 执行失败（退出码 {status}）")

def day_flags(days):
    if days == []:
        raise ValueError("请指定至少一天，或明确设为 None 以运行全量")
    return [value for day in (days or []) for value in ("--day", day)]

run_cli("verify")

## 2. 缓存、事件目录和波谱

这些步骤读取原始 MAG。设置 `EVENT_DAYS` 可先处理少量观测日；改为 `None` 则按事件清单处理全部日期。逐事件结果保存在 `build/raw_catalogue/`，单日或多日诊断写入其 `diagnostic/` 子目录。

In [ ]:
event_flags = ["--events", Path(EVENTS_CSV).expanduser().resolve()] if EVENTS_CSV else []
days = day_flags(EVENT_DAYS)
if RUN_CACHE:
    run_cli("cache", *event_flags, *days, "--workers", WORKERS)
if RUN_CATALOGUE:
    run_cli("catalogue", *event_flags, *days, "--workers", WORKERS)
    output = BUILD / "raw_catalogue"
    if EVENT_DAYS:
        output /= "diagnostic"
    display(pd.json_normalize(read(output / "summary.json")))
if RUN_PSD_AUDIT:
    run_cli("psd-audit", *event_flags, *days, "--workers", WORKERS)
    display(pd.json_normalize(read(BUILD / "raw_psd_summary.json")))

## 3. 构建窗口与训练

使用 BAM 时，先打开 `PREPARE_TRAINING_WINDOWS` 生成所选阶段的训练和验证特征。已有完整特征时可以直接开启 `RUN_TRAIN`。若用冻结特征，将 `TRAIN_INPUT` 设为 `"frozen"` 并提供对应目录。

每个训练任务运行 100 个 epoch。输出目录包含验证记录、选中的检查点及其验证阈值。请为新任务选择尚未使用的阶段和种子；现有训练目录会被拒绝覆盖。`holdout2013` 使用种子 2026。

In [ ]:
from venus_express.repro.model import PROTOCOL

if TRAIN_INPUT not in {"bam", "frozen"}:
    raise ValueError('TRAIN_INPUT 应为 "bam" 或 "frozen"')
if STAGE not in PROTOCOL["training"]:
    raise ValueError(f"未知训练阶段：{STAGE}")

training_root = BUILD / ("training_from_bam" if TRAIN_INPUT == "bam" else "training")

def training_run(seed):
    return training_root / STAGE / f"seed{seed}"

if PREPARE_TRAINING_WINDOWS:
    run_cli("prepare-windows", STAGE)
if RUN_TRAIN:
    for seed in TRAIN_SEEDS:
        flags = ["--from-raw"] if TRAIN_INPUT == "bam" else []
        run_cli("train", STAGE, "--seed", seed, "--device", DEVICE, *flags)
        run = training_run(seed)
        print(f"检查点 {display_path(run / 'selected.pt')}")
        display(pd.read_json(run / "history.json").tail())
        display(pd.DataFrame(read(run / "selection.json")["validation_cuts"]))

## 4. 固定窗口评价

参考评价汇总四阶段各三个种子的结果。默认重算归档分数的指标；使用 `"frozen"` 或 `"bam"` 可用参考检查点重新评分。BAM 评价需要四阶段的验证窗口和固定测试窗口，可用 `PREPARE_EVALUATION_WINDOWS` 统一生成。

设置 `EVAL_TRAINED_SEED` 可额外评价某个本地训练检查点。固定测试阈值取自训练输出中的 80% 验证召回工作点。2013 留出模型请用单独的连续记录评价协议。

In [ ]:
from venusian_pcw.windows import compute as evaluate_reference

if PREPARE_EVALUATION_WINDOWS:
    for stage in ("MSeed", "MExtend1", "MExtend2", "MStandard"):
        run_cli("prepare-windows", stage, "--role", "validation")
    run_cli("prepare-windows", "fixed_test")

if EVALUATION_INPUT not in {None, "cached", "frozen", "bam"}:
    raise ValueError('EVALUATION_INPUT 应为 None、"cached"、"frozen" 或 "bam"')
if EVALUATION_INPUT is not None:
    if EVALUATION_INPUT == "bam":
        run_cli("evaluate-windows", "--device", DEVICE)
        evaluation = read(BUILD / "windows_from_bam" / "summary.json")
    else:
        evaluation = evaluate_reference(rescore=EVALUATION_INPUT == "frozen", device=DEVICE)
    print("参考检查点评价")
    display(pd.DataFrame(evaluation["validation_operating_point"]))
    display(pd.DataFrame(evaluation["equal_test_recall"]))

if EVAL_TRAINED_SEED is not None:
    if STAGE == "holdout2013":
        raise ValueError("请用 2013 留出连续记录协议评价 holdout2013")
    import torch
    from venus_express.repro.model import build_model
    from venus_express.repro.training import arrays, predict
    from venusian_pcw.window_inputs import prepared_path
    from venusian_pcw.windows import measures

    run = training_run(EVAL_TRAINED_SEED)
    selection = read(run / "selection.json")
    cutoff = next(row["logit"] for row in selection["validation_cuts"] if row["recall"] == 0.8)
    rebuilt_test = prepared_path(None, None) if TRAIN_INPUT == "bam" else None
    test = arrays(read(INPUTS / "windows/external_test.json"), rebuilt_test)
    model = build_model()
    model.load_state_dict(torch.load(run / "selected.pt", map_location="cpu", weights_only=True))
    scores = pd.DataFrame(dict(source=test["source"], logit=predict(model.to(DEVICE), test, DEVICE)))
    result = dict(stage=STAGE, seed=EVAL_TRAINED_SEED, threshold_logit=cutoff, **measures(scores, cutoff))
    write(run / "fixed_test_metrics.json", result)
    display(pd.DataFrame([result]))

## 5. 连续搜索与空间曝光

默认日期用于单日验证。将 `SEARCH_DAYS` 改为 `None` 才会扫描所选范围内的全部观测日。搜索用仓库提供的任务期或留出检查点，同一范围的运行共享 `build/raw/` 下的输出。要复用同一次运行的日分数，保持日期、原始数据路径和设备一致，再打开 `RESUME_SEARCH`。

完整空间曝光由 `RUN_EXPOSURE` 单独开启。两条完整搜索完成后，可在 [figures.ipynb](figures.ipynb) 中打开 `USE_NEW_SEARCH`。

In [ ]:
if RUN_SEARCH:
    flags = ["--resume"] if RESUME_SEARCH else []
    run_cli("search", SEARCH_SCOPE, *day_flags(SEARCH_DAYS), "--device", DEVICE, "--workers", WORKERS, *flags)
    complete = read(BUILD / "raw" / SEARCH_SCOPE / "complete.json")
    print(f"已搜索 {len(complete['inputs']['days'])} 天，得到 {complete['parents']} 个父候选。")
if RUN_EXPOSURE:
    run_cli("exposure", "--workers", WORKERS)
    print(display_path(BUILD / "raw" / "exposure"))